# Specialiserede Modeller — 3: CNN — netværk der kan SE

Husker I MNIST-notebookens sidste refleksion? *Netværket ser 784 tal i én lang række og
aner ikke, at pixels har naboer.* I dag indløser vi det løfte: **konvolutionelle neurale
netværk (CNN'er)** er bygget PRÆCIS til at udnytte, at billeder har struktur — og de er
grunden til, at computere i dag kan se.

Dagens data: **FashionMNIST** — 28×28-billeder af tøj i 10 kategorier. Samme format som
MNIST, men markant sværere (prøv selv at skelne en skjorte fra en frakke i 28×28...).

> Denne notebook er selvkørende og kræver kun viden fra Intro-ML — du kan tage emnets notebooks i den rækkefølge, du vil. Der er med vilje flere opgaver, end du kan nå: opgaver mærket **(ekstra)** er til de hurtige, og opgaver mærket **(find fejlen)** indeholder en bevidst fejl, som skal findes og rettes.

## Setup

In [ ]:
!pip install -q kagglehub

# Plottehjælpere + FashionMNIST fra GitHub (Plan B: upload filerne via mappeikonet i Colab)
!wget -q -nc https://raw.githubusercontent.com/UNF-Science-Camps/KIC26/main/98-Helpers/helpers.py
!wget -q -nc https://raw.githubusercontent.com/UNF-Science-Camps/KIC26/main/28-Data/MLData/fashion_traen_lille.csv.gz
!wget -q -nc https://raw.githubusercontent.com/UNF-Science-Camps/KIC26/main/28-Data/MLData/fashion_test_lille.csv.gz

import kagglehub
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F

from helpers import show_images, plot_confusion_matrix

torch.manual_seed(42)
np.random.seed(42)

In [ ]:
train_df = pd.read_csv("fashion_traen_lille.csv.gz")
test_df = pd.read_csv("fashion_test_lille.csv.gz")
print("træning:", train_df.shape, "| test:", test_df.shape)

> **Plan B:** Hvis Kaggle driller, så fjern `#`'erne nedenfor og kør — filerne fra
> vores GitHub er allerede skåret ned, så spring i så fald `sample`-linjerne ovenfor over.

Formatet er præcis som MNIST i Intro-ML: første kolonne er `label`, resten er 784
pixels. Men nu er der en VIGTIG ny detalje: et CNN vil have billederne som **billeder**,
ikke som flade rækker. Shapen skal være `(antal, kanaler, højde, bredde)` — kanaler er
1 for gråtoner (og 3 for RGB-farver, det møder I i -opgaven til sidst):

In [ ]:
class_names = ["T-shirt", "bukser", "sweater", "kjole", "frakke",
               "sandal", "skjorte", "sneaker", "taske", "støvle"]

X_train = torch.tensor(train_df.drop(columns=["label"]).values / 255.0,
                       dtype=torch.float32).reshape(-1, 1, 28, 28)
y_train = torch.tensor(train_df["label"].values, dtype=torch.long)
X_test = torch.tensor(test_df.drop(columns=["label"]).values / 255.0,
                      dtype=torch.float32).reshape(-1, 1, 28, 28)
y_test = torch.tensor(test_df["label"].values, dtype=torch.long)

print("X_traen:", X_train.shape, "  ← (antal, kanaler, højde, bredde)")
show_images(X_train, y_train, n=10, class_names=class_names)

# 1: Konvolution — den glidende linse

## Hvorfor ikke bare et dense-netværk?

I MNIST-notebooken fladede vi billedet ud til 784 tal, og hver eneste pixel fik sin
egen vægt. To problemer:

1. **Naboskab smides væk**: for netværket er pixel 5 og pixel 33 bare to vilkårlige
 tal — det aner ikke, at de sidder lige oven over hinanden.
2. **Intet genbrug**: netværket lærer at genkende en snude i øverste venstre hjørne —
 og aner så INTET, når snuden optræder nederst til højre. Alt skal læres forfra for
 hver position.

## Idéen: en lille kerne, der glider

En **konvolution** løser begge dele. Tag en lille matrix — en **kerne** (fx 2×2 eller
3×3) — og lad den GLIDE hen over billedet. På hver position ganges kernen med det
stykke af billedet, den ligger oven på, og summen skrives i output. Samme kerne,
alle positioner:

In [ ]:
image = torch.tensor([[1., 2., 0., 1.],
                        [0., 1., 3., 1.],
                        [2., 1., 0., 0.],
                        [1., 3., 1., 2.]])

kerne = torch.tensor([[1., 0.],
                      [0., -1.]])

# F.conv2d vil have shapen (antal, kanaler, højde, bredde) — derfor reshape:
result = F.conv2d(image.reshape(1, 1, 4, 4), kerne.reshape(1, 1, 2, 2))
print(result.squeeze())

Tjek det øverste venstre tal i hånden: kernen ligger på $\begin{pmatrix}1 & 2\\0 & 1\end{pmatrix}$,
og $1\cdot 1 + 2\cdot 0 + 0\cdot 0 + 1\cdot(-1) = 0$. Kernen `[[1,0],[0,-1]]` beregner
altså "pixel minus dens diagonale nabo" — den reagerer på ÆNDRINGER på skrå.

Bemærk output-størrelsen: et 4×4-billede og en 2×2-kerne giver 3×3 output — kernen kan
stå på 3×3 forskellige positioner. Generelt: **output = billede − kerne + 1** (uden
padding og stride, mere om dem om lidt).

## Kerner kan SE ting — fx kanter

Her kommer det fede: forskellige kerner opdager forskellige mønstre. Den her berømte
kerne (Sobel-kernen) reagerer kraftigt på **lodrette kanter** — steder hvor billedet
skifter fra lyst til mørkt i vandret retning:

In [ ]:
vertical_edge = torch.tensor([[1., 0., -1.],
                            [2., 0., -2.],
                            [1., 0., -1.]])

shoe = X_train[y_train == 7][0]        # en sneaker fra datasættet

edges = F.conv2d(shoe.reshape(1, 1, 28, 28), vertical_edge.reshape(1, 1, 3, 3))

fig, axes = plt.subplots(1, 2, figsize=(8, 4))
axes[0].imshow(shoe.squeeze(), cmap="gray")
axes[0].set_title("original")
axes[1].imshow(edges.squeeze(), cmap="gray")
axes[1].set_title("lodret-kant-kerne")
for axis in axes:
    axis.axis("off")
plt.show()

Kernen har fremhævet skoens lodrette kanter og ignoreret resten! I et CNN er det
PRÆCIS sådan, de første lag ser: små kerner, der finder kanter, streger og pletter. Den
eneste forskel er, at CNN'et **lærer kernernes tal selv** med gradient descent — i
stedet for at få dem foræret af os.

## Padding, stride og pooling

Tre små begreber, så kan I læse enhver CNN-arkitektur:

- **Padding**: læg en ramme af nuller om billedet, så kernen også kan stå på kanten —
 `padding=1` med en 3×3-kerne bevarer billedstørrelsen.
- **Stride**: lad kernen hoppe flere pixels ad gangen — `stride=2` halverer outputtet.
- **Max-pooling**: skru ned for opløsningen ved at tage MAKSIMUM i små vinduer —
 `MaxPool2d(2)` deler billedet i 2×2-felter og beholder ét tal pr. felt. Det gør
 netværket mindre pixel-pernittengrynet ("der var en kant CIRKA her").

Formlen for output-størrelsen: $\text{ud} = \lfloor(\text{ind} - \text{kerne} + 2\cdot\text{padding})/\text{stride}\rfloor + 1$

In [ ]:
number = torch.tensor([[1., 3., 2., 4.],
                    [5., 2., 1., 0.],
                    [0., 1., 6., 2.],
                    [3., 2., 1., 1.]])

print(F.max_pool2d(number.reshape(1, 1, 4, 4), kernel_size=2).squeeze())

### Opgaver

##### Opgave 1.1
Regn (i hånden!) det MIDTERSTE tal i 3×3-outputtet fra det første eksempel (4×4-billedet
og kernen `[[1,0],[0,-1]]`). Kernen ligger så dens øverste venstre hjørne står på
position (række 1, kolonne 1) — altså på tallet 1 i midten. Tjek dit facit mod
`resultat` ovenfor.

*(Tænk over det, og diskutér med din sidemand — I skal ikke skrive svaret ned.)*

<span style="color:red"><b>LØSNINGSFORSLAG:</b> Kernen ligger på udsnittet [[1, 3], [1, 0]] (rækkerne 1-2, kolonnerne 1-2). Udregningen: 1·1 + 3·0 + 1·0 + 0·(−1) = 1. Og ganske rigtigt: midtertallet i <code>resultat</code> er 1. Konvolution er bare gange-og-læg-sammen på samlebånd.</span>

##### Opgave 1.2
Udfyld det lille billede og den lille kerne, regn ALLE fire output-tal i hånden — og
tjek jer selv med `F.conv2d`. (3×3-billede, 2×2-kerne → 2×2 output.)

In [ ]:
mit_image = torch.tensor([[2., 0., 1.],
                            [1., 3., 0.],
                            [0., 2., 2.]])
min_kerne = torch.tensor([[1., -1.],
                          [0., 1.]])

# øverst venstre:  2·1 + 0·(−1) + 1·0 + 3·1 = 5
# øverst højre:    0·1 + 1·(−1) + 3·0 + 0·1 = −1
# nederst venstre: 1·1 + 3·(−1) + 0·0 + 2·1 = 0
# nederst højre:   3·1 + 0·(−1) + 2·0 + 2·1 = 5

print(F.conv2d(mit_image.reshape(1, 1, 3, 3), min_kerne.reshape(1, 1, 2, 2)).squeeze())

<span style="color:red"><b>LØSNINGSFORSLAG:</b> Facit: [[5, −1], [0, 5]]. Har man regnet ét felt rigtigt, kan man dem alle — resten er samlebåndsarbejde, og det er derfor, computere er så gode til det (og GPU'er endnu bedre: alle felter kan regnes SAMTIDIG).</span>

##### Opgave 1.3
Prøv de to kerner nedenfor på sneaker-billedet (genbrug kant-cellen). Hvad gør de —
og hvorfor giver det mening ud fra tallene i dem?

In [ ]:
identity = torch.tensor([[0., 0., 0.],
                          [0., 1., 0.],
                          [0., 0., 0.]])
smear = torch.ones(3, 3) / 9

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
axes[0].imshow(shoe.squeeze(), cmap="gray")
axes[0].set_title("original")
for axis, (name, k) in zip(axes[1:], [("identitet", identity), ("udtværing", smear)]):
    ud = F.conv2d(shoe.reshape(1, 1, 28, 28), k.reshape(1, 1, 3, 3))
    axis.imshow(ud.squeeze(), cmap="gray")
    axis.set_title(name)
for axis in axes:
    axis.axis("off")
plt.show()

<span style="color:red"><b>LØSNINGSFORSLAG:</b> Identitets-kernen kopierer bare midterpixlen — billedet er uændret (minus en 1-pixel kant). 1/9-kernen erstatter hver pixel med GENNEMSNITTET af sig selv og de 8 naboer — det er en blur/udtværing (og faktisk præcis hvad "blur" gør i billedredigering — nu ved I hvordan Instagram-filtre virker indeni).</span>

##### Opgave 1.4
Sobel-kernen fandt LODRETTE kanter. Byg dens søster, der finder VANDRETTE kanter (vip
tallene 90 grader — eller brug `.T`, transponering), og kør den på sneakeren og på en
taske (`X_traen[y_traen == 8][0]`). Sammenlign de to kant-billeder.

In [ ]:
horizontal_edge = torch.tensor([[1., 2., 1.],
                             [0., 0., 0.],
                             [-1., -2., -1.]])
# (samme som lodret_kant.T)

taske = X_train[y_train == 8][0]
fig, axes = plt.subplots(2, 2, figsize=(8, 8))
for row, image in enumerate([shoe, taske]):
    for kolonne, k in enumerate([vertical_edge, horizontal_edge]):
        ud = F.conv2d(image.reshape(1, 1, 28, 28), k.reshape(1, 1, 3, 3))
        axes[row, kolonne].imshow(ud.squeeze(), cmap="gray")
        axes[row, kolonne].axis("off")
axes[0, 0].set_title("lodret-kant")
axes[0, 1].set_title("vandret-kant")
plt.show()

<span style="color:red"><b>LØSNINGSFORSLAG:</b> Vandret-kanten er lodret-kernen transponeret: positive tal foroven, negative forneden — den reagerer, hvor billedet skifter lyshed i LODRET retning, dvs. på vandrette kanter (skosålen lyser op!). To kerner = to forskellige "spørgsmål" til billedet — et CNN stiller typisk 16-64 af den slags spørgsmål i sit første lag.</span>

##### Opgave 1.5 (find fejlen)
Koden vil køre en kerne på et billede, men PyTorch protesterer over dimensionerne. Læs
fejlbeskeden og fix koden. (Hvilken shape kræver `F.conv2d` — og hvad har `sko` frisk
fra datasættet?)

In [ ]:
kerne = torch.tensor([[1., -1.]])
ud = F.conv2d(shoe.reshape(1, 1, 28, 28), kerne.reshape(1, 1, 1, 2))
print(ud.shape)

<span style="color:red"><b>LØSNINGSFORSLAG:</b> <code>F.conv2d</code> kræver 4D: (antal, kanaler, højde, bredde) — for BÅDE billede og kerne. <code>sko.squeeze()</code> er 2D (28, 28), og kernen var også 2D. Løsningen er to reshapes. 4D-kravet ser pedantisk ud på ét billede, men det er det, der lader PyTorch køre 64 billeder × 32 kerner i ét hug.</span>

##### Opgave 1.6
Kør 3×3-kernen på sneakeren med `padding=0`, `padding=1` og `padding=5` (ekstra
argument til `F.conv2d`). Tjek output-shapen hver gang — og kig på billedet ved
padding=5: hvad er det for en sort ramme?

In [ ]:
for p in [0, 1, 5]:
    ud = F.conv2d(shoe.reshape(1, 1, 28, 28), vertical_edge.reshape(1, 1, 3, 3), padding=p)
    print(f"padding={p}: output {tuple(ud.shape)}")
    plt.imshow(ud.squeeze(), cmap="gray")
    plt.title(f"padding = {p}")
    plt.axis("off")
    plt.show()

<span style="color:red"><b>LØSNINGSFORSLAG:</b> padding=0 → 26×26 (billedet krymper), padding=1 → 28×28 (størrelsen bevares — derfor er padding=1 standard med 3×3-kerner), padding=5 → 36×36 med en tydelig sort ramme: det ER nul-polstringen, man kan se. Kernen har bare regnet videre ud i nullerne.</span>

##### Opgave 1.7
Udfyld formlen $\text{ud} = (\text{ind} - \text{kerne} + 2\cdot\text{padding})/\text{stride} + 1$
som Python-funktion, og tjek den mod PyTorch i de tre tilfælde.

In [ ]:
def output_size(ind, kerne, padding, stride):
    return (ind - kerne + 2 * padding) // stride + 1

for kerne_str, padding, stride in [(3, 0, 1), (3, 1, 2), (5, 2, 2)]:
    beregnet = output_size(28, kerne_str, padding, stride)
    faktisk = F.conv2d(shoe.reshape(1, 1, 28, 28),
                       torch.ones(1, 1, kerne_str, kerne_str),
                       padding=padding, stride=stride).shape[-1]
    print(f"kerne {kerne_str}, padding {padding}, stride {stride}: formel {beregnet}, PyTorch {faktisk}")

<span style="color:red"><b>LØSNINGSFORSLAG:</b> <code>(ind - kerne + 2*padding) // stride + 1</code> — heltalsdivision, fordi en halv position ikke tæller. Facit: 26, 14, 14. Denne formel skal I bruge, hver gang I ændrer i en CNN-arkitektur (og glemmer I den, minder PyTorch's shape-fejl jer venligt(?) om det — se opgave 2.4).</span>

##### Opgave 1.8
Kør max-pooling (`F.max_pool2d(..., kernel_size=2)`) på kant-billedet fra
Sobel-eksemplet, og derefter ENDNU en gang på resultatet. Kig på de tre billeder —
hvad overlever poolingen, og hvad forsvinder?

In [ ]:
edges = F.conv2d(shoe.reshape(1, 1, 28, 28), vertical_edge.reshape(1, 1, 3, 3))
pooled = F.max_pool2d(edges, kernel_size=2)
pooled2 = F.max_pool2d(pooled, kernel_size=2)

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
for axis, (name, b) in zip(axes, [("kanter 26×26", edges), ("poolet 13×13", pooled), ("poolet igen 6×6", pooled2)]):
    axis.imshow(b.squeeze(), cmap="gray")
    axis.set_title(name)
    axis.axis("off")
plt.show()

<span style="color:red"><b>LØSNINGSFORSLAG:</b> De STÆRKE kant-signaler overlever (max tager jo det største tal i hvert felt), mens den præcise pixel-position forsvinder — 6×6-billedet siger "der er en kraftig lodret kant cirka HER", ikke "på pixel (14, 9)". Det er præcis den slags robusthed, man vil have: en sko er en sko, selvom den står to pixels længere til venstre.</span>

##### Opgave 1.9
Dense-laget lærer én vægt pr. pixel pr. neuron. Konvolutionslaget genbruger de samme
9 tal (3×3-kernen) over HELE billedet. Giv to grunde til, at genbruget er smart —
tænk på (1) antal parametre og (2) hvad der sker, når motivet flytter sig i billedet.

*(Tænk over det, og diskutér med din sidemand — I skal ikke skrive svaret ned.)*

<span style="color:red"><b>LØSNINGSFORSLAG:</b> (1) Parametre: en 3×3-kerne er 9 tal (plus bias), uanset billedstørrelse — et dense-lag fra 784 pixels til bare 16 neuroner er 12.560 tal. (2) Flytning: kernen glider jo over ALLE positioner, så en kant genkendes uanset hvor den står — dense-nettet skal lære hver position separat. Det kaldes translations-robusthed, og I måler den selv i opgave 2.6!</span>

##### Opgave 1.10 (ekstra)
Design jeres egen kerne, der reagerer på **diagonale** kanter (fra øverst-venstre mod
nederst-højre). Test den på sneakeren og tasken — og find på mindst ét tjek, der
overbeviser jer om, at den gør det, I tror.

In [ ]:
diagonal_kerne = torch.tensor([[ 2.,  1.,  0.],
                               [ 1.,  0., -1.],
                               [ 0., -1., -2.]])

# tjek på et KUNSTIGT billede med en kendt diagonal:
kunstigt = torch.zeros(28, 28)
for i in range(28):
    kunstigt[i, :i] = 1.0        # trekant → én lang diagonal kant

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
axes[0].imshow(kunstigt, cmap="gray")
axes[0].set_title("kunstig diagonal")
axes[1].imshow(F.conv2d(kunstigt.reshape(1, 1, 28, 28), diagonal_kerne.reshape(1, 1, 3, 3)).squeeze(), cmap="gray")
axes[1].set_title("kernens reaktion")
axes[2].imshow(F.conv2d(shoe.reshape(1, 1, 28, 28), diagonal_kerne.reshape(1, 1, 3, 3)).squeeze(), cmap="gray")
axes[2].set_title("sneakeren")
for axis in axes:
    axis.axis("off")
plt.show()

<span style="color:red"><b>LØSNINGSFORSLAG:</b> Opskriften: positive tal på den ene side af diagonalen, negative på den anden, og ~0 på selve diagonalen. Det stærkeste tjek er et kunstigt billede med en kendt diagonal kant — lyser kernens output op langs den, virker den. At OPFINDE et tjek er i øvrigt halvdelen af al ML-fejlfinding.</span>

# 2: Byg et CNN — og lad det lære kernerne selv

Nu samler vi klodserne. Et klassisk lille CNN ser sådan ud:

```
billede (1×28×28)
  → Conv2d(1→16 kerner, 3×3, padding=1) → ReLU → MaxPool(2)     "find 16 slags småmønstre, zoom ud"
  → Conv2d(16→32 kerner, 3×3, padding=1) → ReLU → MaxPool(2)    "kombinér til 32 større mønstre, zoom ud"
  → flad ud → Linear(32·7·7 → 10)                               "afgør: hvilken tøjtype?"
```

Bemærk logikken: konvolutionslagene finder mønstre (kanter → hjørner → ærmer → sko-form),
poolingen zoomer gradvist ud, og til allersidst samler ét dense-lag det hele til en
beslutning. Størrelserne undervejs: 28 → 28 → 14 → 14 → 7 (padding=1 bevarer størrelsen,
poolingen halverer — regn selv efter med formlen fra 1.7!).

Og træningsloopet? **Fuldstændig uændret** fra Intro-ML — CrossEntropyLoss, Adam,
mini-batches. Et CNN er bare et `nn.Module` med andre lag indeni:

In [ ]:
class ClothesNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 16, kernel_size=3, padding=1)    # 1 kanal ind → 16 kerner
        self.conv2 = nn.Conv2d(16, 32, kernel_size=3, padding=1)   # 16 ind → 32 kerner
        self.pool = nn.MaxPool2d(2)
        self.activation = nn.ReLU()
        self.fc = nn.Linear(32 * 7 * 7, 10)              # 32 kort á 7×7 → 10 klasser

    def forward(self, x):
        x = self.pool(self.activation(self.conv1(x)))    # 28 → 28 → 14
        x = self.pool(self.activation(self.conv2(x)))    # 14 → 14 → 7
        x = x.reshape(x.shape[0], -1)                    # flad ud: (N, 1568)
        return self.fc(x)                                # rå point (CrossEntropy-reglen!)

model = ClothesNet()
print(model)
print("parametre:", sum(p.numel() for p in model.parameters()))

In [ ]:
import time

torch.manual_seed(42)
model = ClothesNet()
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

start = time.time()
for epoch in range(5):
    for i in range(0, len(X_train), 64):
        optimizer.zero_grad()
        loss = loss_fn(model(X_train[i:i + 64]), y_train[i:i + 64])
        loss.backward()
        optimizer.step()
    print(f"epoke {epoch + 1}: tab på sidste portion = {loss.item():.4f}")
print(f"træningstid: {time.time() - start:.0f} s")

In [ ]:
with torch.no_grad():
    pred = model(X_test).argmax(dim=1)
print(f"CNN test-accuracy: {(pred == y_test).float().mean().item():.1%}")

## Duellen: CNN mod dense-netværk

Er al konvolutions-besværet det værd? Lad os træne jeres velkendte dense-net fra
Intro-ML på præcis samme data — og sammenligne både accuracy OG antal parametre:

In [ ]:
torch.manual_seed(42)
dense_model = nn.Sequential(nn.Flatten(),                 # (N,1,28,28) → (N,784)
                            nn.Linear(784, 128), nn.ReLU(),
                            nn.Linear(128, 10))
optimizer = torch.optim.Adam(dense_model.parameters(), lr=0.001)

for epoch in range(5):
    for i in range(0, len(X_train), 64):
        optimizer.zero_grad()
        loss = loss_fn(dense_model(X_train[i:i + 64]), y_train[i:i + 64])
        loss.backward()
        optimizer.step()

with torch.no_grad():
    dense_pred = dense_model(X_test).argmax(dim=1)

print(f"CNN:   {(pred == y_test).float().mean().item():.1%}  med {sum(p.numel() for p in model.parameters()):>7} parametre")
print(f"dense: {(dense_pred == y_test).float().mean().item():.1%}  med {sum(p.numel() for p in dense_model.parameters()):>7} parametre")

CNN'et vinder — med ca. **5× færre parametre**. Struktur-viden slår rå størrelse.
(Og forspringet vokser med mere data og træning: på det fulde FashionMNIST når denne
arkitektur ~90 %+, og på rigtige fotos er dense-net helt chanceløse.)

## Hvor tager modellen fejl? Forvirringsmatricen

Accuracy er ét tal — **forvirringsmatricen** viser HVILKE klasser der forveksles:

In [ ]:
plot_confusion_matrix(y_test, pred, class_names=class_names)

Kig på de store tal UDEN FOR diagonalen: skjorte ↔ T-shirt ↔ frakke ↔ sweater
forveksles flittigt (fair nok — i 28×28 gråtoner!), mens bukser og tasker næsten aldrig
rammes forkert.

## Kig ind i maskinen: de lærte kerner

Vi håndbyggede kant-kerner i afsnit 1 — hvad har NETVÆRKET selv fundet på? `conv1` har
16 kerner à 3×3, og vi kan også se **feature maps**: hvad hver kerne "ser" i et
konkret billede:

In [ ]:
kerner = model.conv1.weight.detach()          # shape (16, 1, 3, 3)

fig, axes = plt.subplots(2, 8, figsize=(12, 3.5))
for i, axis in enumerate(axes.ravel()):
    axis.imshow(kerner[i, 0], cmap="gray")
    axis.set_title(f"kerne {i}", fontsize=8)
    axis.axis("off")
plt.suptitle("De 16 lærte kerner i conv1")
plt.show()

In [ ]:
with torch.no_grad():
    feature_maps = model.activation(model.conv1(shoe.reshape(1, 1, 28, 28)))

fig, axes = plt.subplots(2, 8, figsize=(12, 3.5))
for i, axis in enumerate(axes.ravel()):
    axis.imshow(feature_maps[0, i], cmap="gray")
    axis.axis("off")
plt.suptitle("Hvad de 16 kerner ser i sneakeren (feature maps)")
plt.show()

### Opgaver

##### Opgave 2.1
Brug `vis_billeder` til at se 10 billeder, som CNN'et gætter FORKERT (mønstret kender I
fra Intro-ML: `(gaet!= y_test).nonzero()`). Passer fejlene med forvirringsmatricens
værste par — og er der nogen, hvor I selv ville have gættet det samme som modellen?

In [ ]:
forkerte = (pred != y_test).nonzero().squeeze()
print("antal fejl:", len(forkerte))

show_images(X_test[forkerte[:10]], y_test[forkerte[:10]],
             predictions=pred[forkerte[:10]], n=10, class_names=class_names)

<span style="color:red"><b>LØSNINGSFORSLAG:</b> Langt de fleste fejl er netop skjorte/T-shirt/frakke/sweater-forvekslinger — og i 28×28 gråtoner er mange af dem reelt umulige (mennesker scorer selv kun ~83-87 % på FashionMNIST uden zoom!). Modellens fejl er med andre ord for det meste RIMELIGE fejl — det er et sundhedstegn.</span>

##### Opgave 2.2
Skru op for antallet af kerner: prøv (32, 64) i stedet for (16, 32) — husk at `fc`-lagets
input så også ændrer sig! Hvad sker der med accuracy og træningstid?

In [ ]:
class BigClothesNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 32, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.pool = nn.MaxPool2d(2)
        self.activation = nn.ReLU()
        self.fc = nn.Linear(64 * 7 * 7, 10)

    def forward(self, x):
        x = self.pool(self.activation(self.conv1(x)))
        x = self.pool(self.activation(self.conv2(x)))
        x = x.reshape(x.shape[0], -1)
        return self.fc(x)

torch.manual_seed(42)
stort = BigClothesNet()
optimizer = torch.optim.Adam(stort.parameters(), lr=0.001)
start = time.time()
for epoch in range(5):
    for i in range(0, len(X_train), 64):
        optimizer.zero_grad()
        loss = loss_fn(stort(X_train[i:i + 64]), y_train[i:i + 64])
        loss.backward()
        optimizer.step()
with torch.no_grad():
    acc = (stort(X_test).argmax(dim=1) == y_test).float().mean()
print(f"accuracy {acc.item():.1%} på {time.time() - start:.0f} s")

<span style="color:red"><b>LØSNINGSFORSLAG:</b> Målt: ~86,4 % mod ~84,5 % — knap 2 procentpoint bedre, men ~1,5× langsommere og 2,5× flere parametre. På kun 10.000 træningsbilleder er flaskehalsen DATA, ikke modelstørrelse (Intro-ML-lektionen igen). Dét fc-laget skulle ændres til, er 64·7·7 = 3136 — glemte man det, mødte man præcis fejlen fra opgave 2.4.</span>

##### Opgave 2.3
Udfyld parametertællingerne og beregn, hvor mange gange flere parametre dense-nettet
har. Hvor sidder langt de fleste af CNN'ets parametre — i konvolutionslagene eller i
`fc`-laget?

In [ ]:
cnn_total = sum(p.numel() for p in model.parameters())
dense_total = sum(p.numel() for p in dense_model.parameters())

print(f"CNN: {cnn_total}, dense: {dense_total}, forhold: {dense_total / cnn_total:.1f}x")

for name, p in model.named_parameters():
    print(f"{name:15s} {p.numel():>6} tal")

<span style="color:red"><b>LØSNINGSFORSLAG:</b> Dense har 5× flere (101.770 mod 20.490). Og inde i CNN'et bor størstedelen i <code>fc</code>-laget (1568·10 + 10 = 15.690) — de to konvolutionslag er tilsammen under 5.000 tal! Kernerne er ekstremt parameter-billige, fordi de GENBRUGES over hele billedet (opgave 1.9-pointen, nu med tal på).</span>

##### Opgave 2.4 (find fejlen)
Nogen har slettet en linje i `forward` — og nu kaster netværket en lang shape-fejl, når
man kalder det. Kør cellen, læs fejlen, og sæt den manglende linje ind igen. Hvorfor kan
`fc`-laget ikke bare spise conv-outputtet direkte?

In [ ]:
class BrokenNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 16, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(16, 32, kernel_size=3, padding=1)
        self.pool = nn.MaxPool2d(2)
        self.activation = nn.ReLU()
        self.fc = nn.Linear(32 * 7 * 7, 10)

    def forward(self, x):
        x = self.pool(self.activation(self.conv1(x)))
        x = self.pool(self.activation(self.conv2(x)))
        x = x.reshape(x.shape[0], -1)     # ← den manglende linje: flad ud til (N, 1568)
        return self.fc(x)

model_o = BrokenNet()
print(model_o(X_test[:4]).shape)

<span style="color:red"><b>LØSNINGSFORSLAG:</b> Conv-delen afleverer en 4D-tensor (N, 32, 7, 7) — et stakit af feature maps — men <code>nn.Linear</code> spiser kun flade rækker (N, 1568). <code>reshape(x.shape[0], -1)</code> flader alt undtagen batch-dimensionen ud (−1 = "regn selv resten"). Overgangen conv→linear er det klassiske sted, CNN-shapes går i stykker.</span>

##### Opgave 2.5
Tilføj et TREDJE konvolutionslag (32 → 64 kerner, UDEN padding) efter conv2 — og uden
pooling efter (7×7 er allerede småt). Brug formlen fra 1.7 til at regne ud, hvad
`fc`-lagets input skal være, FØR I kører. Blev accuracy bedre?

In [ ]:
class DeepNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 16, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(16, 32, kernel_size=3, padding=1)
        self.conv3 = nn.Conv2d(32, 64, kernel_size=3)
        self.pool = nn.MaxPool2d(2)
        self.activation = nn.ReLU()
        self.fc = nn.Linear(64 * 5 * 5, 10)   # 7 - 3 + 1 = 5 → 64 kort á 5×5

    def forward(self, x):
        x = self.pool(self.activation(self.conv1(x)))
        x = self.pool(self.activation(self.conv2(x)))
        x = self.activation(self.conv3(x))
        x = x.reshape(x.shape[0], -1)
        return self.fc(x)

torch.manual_seed(42)
deep = DeepNet()
optimizer = torch.optim.Adam(deep.parameters(), lr=0.001)
for epoch in range(5):
    for i in range(0, len(X_train), 64):
        optimizer.zero_grad()
        loss = loss_fn(deep(X_train[i:i + 64]), y_train[i:i + 64])
        loss.backward()
        optimizer.step()
with torch.no_grad():
    acc = (deep(X_test).argmax(dim=1) == y_test).float().mean()
print(f"accuracy: {acc.item():.1%}")

<span style="color:red"><b>LØSNINGSFORSLAG:</b> Formlen: 7 − 3 + 1 = 5, så fc-input er 64·5·5 = 1600. Målt: ~83,5 % — altså en anelse DÅRLIGERE end 2-lags-udgavens ~84,5 %! På så små billeder og så lidt data er 2 conv-lag rigeligt, og det ekstra lag giver bare flere parametre at overfitte med. Rigtige billed-net (ResNet m.fl.) har 50+ lag — men også millioner af billeder at lære fra. Dybere er ikke gratis.</span>

##### Opgave 2.6
Robusthedstesten fra opgave 1.9: forskyd ALLE testbilleder 3 pixels til højre med
`torch.roll` og mål både CNN'ets og dense-nettets accuracy på de forskudte billeder.
Hvem tåler flytningen bedst — og hvorfor er ingen af dem uberørt?

In [ ]:
X_test_shifted = torch.roll(X_test, shifts=3, dims=3)

show_images(X_test_shifted, y_test, n=5, class_names=class_names)

with torch.no_grad():
    cnn_acc = (model(X_test_shifted).argmax(dim=1) == y_test).float().mean()
    dense_acc = (dense_model(X_test_shifted).argmax(dim=1) == y_test).float().mean()
print(f"forskudt: CNN {cnn_acc.item():.1%} | dense {dense_acc.item():.1%}")

<span style="color:red"><b>LØSNINGSFORSLAG:</b> Målt: CNN ~43 % mod dense ~30 % — CNN'et tåler flytningen langt bedre (kernerne glider jo og finder mønstrene på den nye position), men er ikke immunt: fc-laget til sidst er stadig positionsafhængigt, og FashionMNIST-billeder er altid centreret, så nettet har aldrig SET forskudt tøj. (Kur: træn på tilfældigt forskudte kopier — det hedder data augmentation og er standard i alle rigtige billed-pipelines.)</span>

##### Opgave 2.7
Udfyld softmax-kaldet og find det billede i testsættet, hvor modellen er MEST i tvivl
(mindste max-sandsynlighed). Vis billedet og dets sandsynligheds-søjler — hvad er
modellen i tvivl imellem?

In [ ]:
with torch.no_grad():
    probabilities = torch.softmax(model(X_test), dim=1)

max_probability = probabilities.max(dim=1).values
i = max_probability.argmin().item()

fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
axes[0].imshow(X_test[i].squeeze(), cmap="gray")
axes[0].set_title(f"label: {class_names[y_test[i]]}")
axes[0].axis("off")
axes[1].bar(class_names, probabilities[i])
axes[1].tick_params(axis="x", rotation=45)
plt.show()

<span style="color:red"><b>LØSNINGSFORSLAG:</b> <code>dim=1</code> (klasse-dimensionen). Det mest usikre billede er typisk et grumset stykke overtøj, hvor sandsynligheden er smurt ud over skjorte/frakke/sweater med ~25-35 % til hver. En models tvivl er værdifuld information — i produktion sendes den slags til et menneske.</span>

##### Opgave 2.8
Feature maps-figuren viser, hvad hver kerne "ser". Kør den med en **taske** og en
**støvle** i stedet for sneakeren. Kan I finde en kerne, der tydeligt er "kant-agtig"
(sammenlign med jeres håndbyggede fra afsnit 1)? Og en, der mest ligner støj?

In [ ]:
image = X_train[y_train == 8][0]      # taske — prøv også 9 (støvle)

with torch.no_grad():
    fm = model.activation(model.conv1(image.reshape(1, 1, 28, 28)))

fig, axes = plt.subplots(2, 8, figsize=(12, 3.5))
for i, axis in enumerate(axes.ravel()):
    axis.imshow(fm[0, i], cmap="gray")
    axis.set_title(f"kerne {i}", fontsize=8)
    axis.axis("off")
plt.show()

<span style="color:red"><b>LØSNINGSFORSLAG:</b> Flere kerner opfører sig tydeligt som kant-detektorer (lyser op langs taskens kontur — sammenlign med Sobel-outputtet fra afsnit 1), mens andre ser udvaskede eller støjede ud — de er enten "døde" (ReLU-lektionen fra Intro-ML!) eller specialiseret i noget, der ikke findes i netop dette billede. At nettet SELV har genopfundet kant-detektion, er en af de smukkeste opdagelser i deep learning — det gjorde hjerneforskerne Hubel & Wiesel i øvrigt også, da de målte på kattehjerner i 1959.</span>

##### Opgave 2.9 (ekstra)
**CIFAR-10: rigtige FARVEBILLEDER.** Koden nedenfor er komplet: den henter CIFAR-10
(fly, biler, katte... — ~170 MB, men Kaggle er hurtig), viser billederne og træner
et farve-CNN (3 input-kanaler!). Kør den, og eksperimentér så: flere epoker? flere
kerner? Hvor langt kan I komme over de ~50 %? (Tilfældig gætning er 10 % — men perfekt
er UMULIGT på 3 epoker og 4.000 billeder.)

In [ ]:
import torchvision

# CIFAR-10 hentes fra Kaggle (hurtigt!) — torchvision læser de udpakkede filer:
path_cifar = kagglehub.dataset_download("pankrzysiu/cifar10-python")
cifar_train = torchvision.datasets.CIFAR10(root=path_cifar, train=True)
cifar_test = torchvision.datasets.CIFAR10(root=path_cifar, train=False)

#  Plan B (hvis Kaggle driller) — hent direkte fra kilden (kan være LANGSOM):
# cifar_traen = torchvision.datasets.CIFAR10(root="cifar_data", train=True, download=True)
# cifar_test = torchvision.datasets.CIFAR10(root="cifar_data", train=False, download=True)

X_c = torch.tensor(cifar_train.data[:4000] / 255.0, dtype=torch.float32).permute(0, 3, 1, 2)
y_c = torch.tensor(cifar_train.targets[:4000], dtype=torch.long)
X_ct = torch.tensor(cifar_test.data[:1000] / 255.0, dtype=torch.float32).permute(0, 3, 1, 2)
y_ct = torch.tensor(cifar_test.targets[:1000], dtype=torch.long)
cifar_names = ["fly", "bil", "fugl", "kat", "hjort", "hund", "frø", "hest", "skib", "lastbil"]
print("X_c:", X_c.shape, " ← 3 kanaler: rød, grøn, blå!")

fig, axes = plt.subplots(1, 6, figsize=(12, 2.5))
for i, axis in enumerate(axes):
    axis.imshow(X_c[i].permute(1, 2, 0))       # tilbage til (H, B, kanaler) for imshow
    axis.set_title(cifar_names[y_c[i]])
    axis.axis("off")
plt.show()

class ColorNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 16, kernel_size=3, padding=1)   # 3 kanaler ind!
        self.conv2 = nn.Conv2d(16, 32, kernel_size=3, padding=1)
        self.pool = nn.MaxPool2d(2)
        self.activation = nn.ReLU()
        self.fc = nn.Linear(32 * 8 * 8, 10)                       # 32 → 16 → 8 med padding=1

    def forward(self, x):
        x = self.pool(self.activation(self.conv1(x)))
        x = self.pool(self.activation(self.conv2(x)))
        x = x.reshape(x.shape[0], -1)
        return self.fc(x)

torch.manual_seed(42)
color_model = ColorNet()
optimizer = torch.optim.Adam(color_model.parameters(), lr=0.001)
loss_fn = nn.CrossEntropyLoss()
for epoch in range(3):
    for i in range(0, len(X_c), 64):
        optimizer.zero_grad()
        loss = loss_fn(color_model(X_c[i:i + 64]), y_c[i:i + 64])
        loss.backward()
        optimizer.step()
    print(f"epoke {epoch + 1} færdig")

with torch.no_grad():
    acc = (color_model(X_ct).argmax(dim=1) == y_ct).float().mean()
print(f"CIFAR-10 accuracy: {acc.item():.1%}  (tilfældigt = 10 %)")

<span style="color:red"><b>LØSNINGSFORSLAG:</b> Efter 3 epoker på 4.000 billeder lander man omkring 40-50 % (målt: 41,7 %) — fire-fem gange bedre end tilfældigt, på rigtige farvefotos, med en model på ~90.000 parametre. Flere epoker/kerner giver et par point mere, men rigtige CIFAR-resultater (95 %+) kræver dybe net, augmentation og GPU-timer. Bemærk de to nye ting i koden: <code>Conv2d(3,...)</code> (rød+grøn+blå ind) og <code>permute</code> (numpy-billeder er (H, B, kanaler), PyTorch vil have (kanaler, H, B)). Lærernote: CIFAR hentes bevidst via kagglehub — det officielle torchvision-mirror (cs.toronto.edu) kan være MEGET langsomt (målt: ~50 KB/s ≈ 1 time), mens Kaggle-kopien (md5-identisk) tager sekunder.</span>

##### Opgave 2.10 (ekstra)
**Det ondeste eksperiment:** omrokér alle 784 pixels med den SAMME tilfældige
permutation i alle billeder (så informationen bevares, men naboskabet ødelægges), og
træn både CNN og dense-net på de omrokerede billeder. Forudsig FØRST: hvem rammes
hårdest — og hvorfor? Kør så og se.

In [ ]:
perm = torch.randperm(784)

def shuffle(X):
    return X.reshape(-1, 784)[:, perm].reshape(-1, 1, 28, 28)

show_images(shuffle(X_train), y_train, n=5, class_names=class_names)

results = {}
for name, build in [("CNN", lambda: ClothesNet()),
                  ("dense", lambda: nn.Sequential(nn.Flatten(), nn.Linear(784, 128),
                                                  nn.ReLU(), nn.Linear(128, 10)))]:
    torch.manual_seed(42)
    m = build()
    optimizer = torch.optim.Adam(m.parameters(), lr=0.001)
    for epoch in range(5):
        for i in range(0, len(X_train), 64):
            optimizer.zero_grad()
            loss = loss_fn(m(shuffle(X_train)[i:i + 64]), y_train[i:i + 64])
            loss.backward()
            optimizer.step()
    with torch.no_grad():
        results[name] = (m(shuffle(X_test)).argmax(dim=1) == y_test).float().mean().item()

for name, acc in results.items():
    print(f"{name}: {acc:.1%} på omrokerede billeder")

<span style="color:red"><b>LØSNINGSFORSLAG:</b> Dense-nettet er (næsten) LIGEGLAD — det så jo alligevel bare 784 løsrevne tal, så en fast omrokering ændrer intet (~84 %, som før). CNN'et derimod mister sin superkraft (falder til ~81 %, altså UNDER dense) — nabopixels er ikke længere naboer, så kanterne, kernerne leder efter, findes ikke mere, og det falder ned omkring dense-niveau eller under. Konklusionen er hele notebookens pointe i ét eksperiment: <b>CNN'ets fordel ER naboskabs-antagelsen.</b> Modeller er gode til det, deres antagelser passer på — det er det, "specialiseret" betyder.</span>